# 🔍 Notebook 04: Daten inspizieren & Explorative Datenanalyse (EDA)

**Szenario**: Du hast einen neuen Datensatz erhalten (flights.csv) – was machst du als erstes?

**Lernziele:**
- df.info(), df.describe(), df.dtypes verstehen
- Fehlende Werte mit missingno visualisieren
- Verteilungen (Histogramm, Boxplot, Violinplot)
- Kategoriale Analysen
- Ausreißer mit Z-Score und IQR erkennen
- Korrelationsanalyse

---

In [ ]:
# ── Setup: Pakete installieren (nur in Colab nötig) ──────────────────────────
import subprocess, sys
packages = ['ydata-profiling', 'missingno', 'plotly', 'kaleido']
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
print("Setup abgeschlossen!")


In [ ]:
BASE_URL = "https://raw.githubusercontent.com/swrobuts/dav/main/data/"


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Stil setzen
sns.set_theme(style='whitegrid', palette='Blues')
plt.rcParams['figure.figsize'] = (10, 6)

df = pd.read_csv(BASE_URL + "flights.csv")
print(f'Dataset geladen: {df.shape[0]:,} Flüge, {df.shape[1]} Spalten')

## 1️⃣ Erster Blick: head(), tail(), info(), describe()

In [ ]:
print('=== df.head(5) ===')
print(df.head(5))
print()
print('=== df.dtypes ===')
print(df.dtypes)
print()
print('=== df.info() ===')
df.info()

In [ ]:
print('=== df.describe() – Numerische Spalten ===')
print(df.describe().round(2))
print()
print('=== df.describe(include="object") – Kategorische Spalten ===')
print(df.describe(include='object'))

## 2️⃣ Fehlende Werte analysieren

In [ ]:
print('=== Fehlende Werte (absolut) ===')
print(df.isnull().sum())
print()
print('=== Fehlende Werte (prozentual) ===')
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

In [ ]:
try:
    import missingno as msno
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    msno.matrix(df, ax=axes[0], color=(0.22, 0.54, 0.81), sparkline=False)
    axes[0].set_title('Missing Values Matrix', fontsize=12, fontweight='bold')
    msno.bar(df, ax=axes[1], color='#E8792F')
    axes[1].set_title('Datenvollständigkeit je Spalte', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('missingno nicht verfügbar – Alternative:')
    missing = df.isnull().sum()
    missing[missing > 0].plot(kind='bar', color='#E8792F', figsize=(10,4))
    plt.title('Fehlende Werte je Spalte')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 3️⃣ Verteilungsanalyse

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Histogramm: Departure Delay
delays = df['dep_delay'].dropna()
axes[0,0].hist(delays, bins=50, color='#0389CF', alpha=0.7, edgecolor='white')
axes[0,0].set_title('Departure Delay – Histogramm', fontweight='bold')
axes[0,0].set_xlabel('Verspätung (Minuten)')
axes[0,0].set_ylabel('Häufigkeit')
axes[0,0].axvline(delays.mean(), color='#E8792F', linestyle='--', label=f'Mean: {delays.mean():.1f}')
axes[0,0].legend()

# Boxplot: Delays nach Airline
top_carriers = df['carrier'].value_counts().head(8).index
df_top = df[df['carrier'].isin(top_carriers)].dropna(subset=['dep_delay'])
df_top.boxplot(column='dep_delay', by='carrier', ax=axes[0,1])
axes[0,1].set_title('Delay je Airline – Boxplot')
axes[0,1].set_suptitle = ''
plt.suptitle('')
axes[0,1].set_xlabel('Airline')

# Violinplot
sns.violinplot(data=df[df['dep_delay'].between(-30, 120)], y='dep_delay', ax=axes[1,0],
               color='#0389CF')
axes[1,0].set_title('Departure Delay – Violinplot')

# Kategorisch: Flüge je Airline
carrier_counts = df['carrier'].value_counts().head(10)
axes[1,1].bar(carrier_counts.index, carrier_counts.values, color='#0389CF')
axes[1,1].set_title('Flüge je Airline (Top 10)')
axes[1,1].set_xlabel('Airline')
axes[1,1].set_ylabel('Anzahl')

plt.tight_layout()
plt.savefig("/tmp/eda_distributions.png", dpi=100, bbox_inches='tight')
plt.show()

## 4️⃣ Ausreißer erkennen: Z-Score & IQR

In [ ]:
from scipy import stats

delays_clean = df['dep_delay'].dropna()

# Z-Score
z_scores = np.abs(stats.zscore(delays_clean))
outliers_z = delays_clean[z_scores > 3]
print(f'Z-Score Ausreißer (|z|>3): {len(outliers_z):,} ({len(outliers_z)/len(delays_clean)*100:.1f}%)')
print(f'  Min: {outliers_z.min():.0f} min, Max: {outliers_z.max():.0f} min')

# IQR
Q1 = delays_clean.quantile(0.25)
Q3 = delays_clean.quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers_iqr = delays_clean[(delays_clean < lower) | (delays_clean > upper)]
print(f'\nIQR Ausreißer: {len(outliers_iqr):,} ({len(outliers_iqr)/len(delays_clean)*100:.1f}%)')
print(f'  Grenzen: [{lower:.1f}, {upper:.1f}] Minuten')
print(f'\n💡 Fazit: ~{len(outliers_iqr)/len(delays_clean)*100:.0f}% der Flüge haben extreme Verspätungen')

## 5️⃣ Korrelationsanalyse

In [ ]:
# Korrelationsmatrix
numeric_df = df.select_dtypes(include='number').dropna()
corr = numeric_df.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Korrelationsmatrix – Flugdaten', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Stärkste Korrelationen
print('\nTop 5 stärkste Korrelationen (außer Selbst-Korrelation):')
corr_pairs = corr.unstack()
corr_pairs = corr_pairs[corr_pairs < 0.999]
print(corr_pairs.abs().sort_values(ascending=False).head(10).to_string())

## ✅ Challenges

1. Welche Airline hat die höchste durchschnittliche Ankunftsverspätung?
2. Gibt es einen Unterschied zwischen Montags- und Freitagsflügen bei der Verspätung?
3. Visualisiere die Verteilung der Fluglänge (distance) als Histogramm

In [ ]:
# Challenge 1:

# Challenge 2:

# Challenge 3:


---
**Weiter mit:** `05_Daten_Bereinigen.ipynb`